# 欢迎来到 RAG 周！！

## 专家知识工作者

### 一个问答助手，扮演专家知识工作者
### 供保险科技公司 Insurellm 的员工使用
### AI 助手需要准确，且方案应低成本。

本项目将使用 RAG（Retrieval Augmented Generation，检索增强生成），确保我们的问答助手具有高准确率。

这第一个实现会使用一种简单、蛮力式的 RAG……

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">本周项目的商业应用</h2>
            <span style="color:#181;">RAG 或许是本课程所覆盖内容中最能立即落地的技术！事实上，已有商业产品正是在做我们本周要构建的事：在大型信息库（如公司合同或产品规格）上进行细致查询。RAG 为你提供了一种快速上市、低成本的机制，让 LLM 适配你的业务领域。</span>
        </td>
    </tr>
</table>

In [ ]:
# 导入所需库：os/glob 读文件，dotenv 加载密钥，
# Path 处理路径，gradio 做聊天界面，OpenAI 调用大模型

import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [ ]:
# 环境设置

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

### 让我们把所有员工数据读入一个字典

In [ ]:
# 把员工知识库读入字典 knowledge（蛮力式 RAG 的数据源）
# 键 = 姓氏小写，值 = 文件全文

knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
# 查看整个 knowledge 字典

knowledge

In [ ]:
# 用姓氏 lancaster 取出对应员工文档

knowledge["lancaster"]

In [ ]:
# 同样方式加载产品文档，并入同一个 knowledge 字典

filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
# 查看 knowledge 里有哪些键（员工姓 / 产品名）

knowledge.keys()

In [ ]:
# 系统提示词前缀：告诉模型它代表 Insurellm，并预留「相关上下文」位置
# （后面会把 retrieve 到的文档拼到 Relevant context 后面）

SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
# 简单检索（retrieve）：从用户问题里抽出单词，
# 若单词恰好是 knowledge 的键，就把对应文档当作相关上下文
# 这是最简陋的 RAG——没有 embedding，只靠关键词匹配

def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_context = []
    for word in words:
        if word in knowledge:
            relevant_context.append(knowledge[word])
    return relevant_context          

## 但更 Pythonic 的写法：

In [ ]:
# 更紧凑的写法：列表推导实现同样的关键词检索

def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]   

In [ ]:
# 试一试：问 lancaster，应命中员工文档

get_relevant_context("Who is lancaster?")

In [ ]:
# 问两个实体：Lancaster（员工）和 carllm（产品）

get_relevant_context("Who is Lancaster and what is carllm?")

In [ ]:
# 把检索到的文档拼成一段「附加上下文」字符串，准备塞进系统提示词
# 若什么都没检索到，就明确告诉模型没有相关上下文

def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [ ]:
# 打印看看检索+拼接后的附加上下文长什么样

print(additional_context("Who is Alex Lancaster?"))

In [ ]:
# RAG 聊天核心：system = 角色说明 + 检索到的上下文，再加历史与用户问题，调用 LLM

def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## 现在我们用 Gradio 的 Chat 界面把它呈现出来 —

一种快速、简便的方式来与 LLM 做聊天原型

In [ ]:
# 用 Gradio 启动聊天界面；type="messages" 表示多轮消息格式

view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)